# Sales Forecasting Experiments — Prophet + MLflow

**Purpose.** This notebook trains the current forecasting model (**Prophet**) on the current
dataset and logs every run to **MLflow** so the team can tune hyperparameters and compare results.

**How to use it**
1. Select the project's **`.venv` (Python 3.11)** kernel — that is the only environment where
   `prophet`, `mlflow`, and `sklearn` are installed.
2. Edit the values in the **`HYPERPARAMS`** cell (marked ★ EDIT THESE).
3. Run all cells. Each run is logged to MLflow as a new entry.
4. Compare runs in the MLflow UI:
   ```
   .venv\Scripts\python -m mlflow ui --backend-store-uri sqlite:///mlflow.db
   ```
   then open http://localhost:5000 and look at the **`olist_forecasting`** experiment.

> This notebook is **self-contained** — it does not import from `ml_engine/` or `core/`. It mirrors
> the logic in `ml_engine/forecasting.py` so results match the production script.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.prophet
from prophet import Prophet

## Paths & MLflow setup
Resolves the project root automatically whether you run from the project root or the `notebooks/`
folder, and points MLflow at the same `mlflow.db` the scripts use.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data Cleaning + EDA").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "Data Cleaning + EDA" / "cleaned_master_df.parquet"
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("olist_forecasting")

print("Project root :", PROJECT_ROOT)
print("Data path    :", DATA_PATH, "(exists:", DATA_PATH.exists(), ")")
print("MLflow store :", MLFLOW_DB)

## Load data & build the daily sales series
Aggregates the cleaned master table to one row per day, then trims the sparse ramp-up (late 2016)
and the incomplete data-collection tail (late Aug 2018) — identical to `build_daily_forecasting_df`
in `data/pipeline.py`.

In [ ]:
master = pd.read_parquet(DATA_PATH)
master["order_purchase_timestamp"] = pd.to_datetime(master["order_purchase_timestamp"], errors="coerce")

daily = (
    master
    .groupby(master["order_purchase_timestamp"].dt.normalize())
    .agg(
        total_sales=("payment_value", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_unique_id", "nunique"),
        avg_freight=("freight_value", "mean"),
    )
    .reset_index()
    .rename(columns={"order_purchase_timestamp": "date"})
)
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").reset_index(drop=True)

# Trim sparse leading/trailing days (< 20% of median daily orders)
if len(daily) > 30:
    threshold = daily["total_orders"].median() * 0.2
    dense = daily.index[daily["total_orders"] >= threshold]
    if len(dense) > 0:
        daily = daily.loc[dense.min():dense.max()].reset_index(drop=True)

print(f"Daily series: {len(daily)} days  ({daily['date'].min().date()} to {daily['date'].max().date()})")
daily.head()

## ★ HYPERPARAMETERS — EDIT THESE
These are the **current production defaults**. Change any value, then **Run All** to log a new
MLflow run you can compare against previous ones.

| Param | Meaning |
|---|---|
| `seasonality_mode` | `"multiplicative"` or `"additive"` |
| `changepoint_prior_scale` | trend flexibility — higher = more flexible |
| `seasonality_prior_scale` | strength of seasonality |
| `holidays_prior_scale` | strength of holiday effects |
| `yearly_seasonality` / `weekly_seasonality` | toggle seasonal components |
| `forecast_periods` | days to forecast into the future for the plot |

In [ ]:
HYPERPARAMS = {
    "seasonality_mode":        "multiplicative",
    "changepoint_prior_scale": 0.05,
    "seasonality_prior_scale": 10.0,
    "holidays_prior_scale":    10.0,
    "yearly_seasonality":      True,
    "weekly_seasonality":      True,
}

forecast_periods = 30  # days to project beyond history (for the forecast plot)

## Train / test split (chronological 80/20)
Time-series data must be split in time order — the most recent 20% is held out for evaluation.

In [ ]:
prophet_df = daily.rename(columns={"date": "ds", "total_sales": "y"})[["ds", "y"]].copy()
prophet_df["ds"] = pd.to_datetime(prophet_df["ds"])
prophet_df = prophet_df.dropna().sort_values("ds").reset_index(drop=True)

split = int(len(prophet_df) * 0.8)
train_df = prophet_df.iloc[:split]
test_df  = prophet_df.iloc[split:]

print(f"Train: {len(train_df)} days   Test: {len(test_df)} days")

## Train Prophet & log to MLflow
Logs the hyperparameters, the evaluation metrics (**MAE / RMSE / MAPE**), and the fitted model as
one MLflow run.

In [ ]:
def evaluate(model, test_df):
    """MAE / RMSE / MAPE on the held-out test window (mirrors forecasting.py)."""
    future = model.make_future_dataframe(periods=len(test_df), freq="D")
    fc = model.predict(future)
    predicted = fc.tail(len(test_df))["yhat"].values
    actual = test_df["y"].values
    mae  = float(np.mean(np.abs(actual - predicted)))
    rmse = float(np.sqrt(np.mean((actual - predicted) ** 2)))
    mape = float(np.mean(np.abs((actual - predicted) / (actual + 1e-9))) * 100)
    return {"mae": mae, "rmse": rmse, "mape": mape}


with mlflow.start_run() as run:
    params = {**HYPERPARAMS, "train_rows": len(train_df), "test_rows": len(test_df)}
    mlflow.log_params(params)

    model = Prophet(**HYPERPARAMS)
    model.fit(train_df)

    metrics = evaluate(model, test_df)
    mlflow.log_metrics(metrics)

    mlflow.prophet.log_model(model, artifact_path="prophet_model")

    run_id = run.info.run_id

print(f"MLflow run {run_id[:8]} logged to 'olist_forecasting'")
print(f"MAE = {metrics['mae']:,.2f}   RMSE = {metrics['rmse']:,.2f}   MAPE = {metrics['mape']:.2f}%")

## Visualize the forecast
Historical actuals + the forecast with its confidence band, and the actual-vs-predicted comparison
on the held-out test window.

In [ ]:
future = model.make_future_dataframe(periods=forecast_periods, freq="D")
forecast = model.predict(future)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(prophet_df["ds"], prophet_df["y"], color="#4F46E5", lw=1.2, label="Actual sales")
ax.plot(forecast["ds"], forecast["yhat"], color="#94A3B8", lw=1.5, ls="--", label="Forecast")
ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                color="#4F46E5", alpha=0.12, label="Confidence band")
ax.axvline(test_df["ds"].iloc[0], color="#DC2626", ls=":", lw=1, label="Train/test split")
ax.set_title("Prophet forecast vs actual daily sales")
ax.set_xlabel("Date"); ax.set_ylabel("Total sales (R$)"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Actual vs predicted on the test window
test_future = model.make_future_dataframe(periods=len(test_df), freq="D")
test_fc = model.predict(test_future).tail(len(test_df))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(test_df["ds"].values, test_df["y"].values, color="#4F46E5", lw=1.5, label="Actual")
ax.plot(test_df["ds"].values, test_fc["yhat"].values, color="#F59E0B", lw=1.5, label="Predicted")
ax.set_title("Test window — actual vs predicted")
ax.set_xlabel("Date"); ax.set_ylabel("Total sales (R$)"); ax.legend()
plt.tight_layout(); plt.show()

## Compare your runs
Every time you edit `HYPERPARAMS` and re-run, a new run appears under the `olist_forecasting`
experiment. Launch the UI and sort by `mape` (or `rmse`) to find the best configuration:

```
.venv\Scripts\python -m mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Open http://localhost:5000 → experiment **olist_forecasting** → select runs → **Compare**.

> **Note on MAPE:** daily Olist sales are volatile and grew ~10x over the period, so MAPE is
> naturally high (~65%). Compare runs *relative to each other*; RMSE/MAE are more stable to read.